
# Optimized BioCatch Vishing Data Augmentation Pipeline

Versión optimizada para ejecución en Kaggle P100/T4 con enfoque en:

- reducción de consumo RAM
- generación chunked
- exportación incremental
- entrenamiento progresivo
- control de memoria
- parquet streaming
- validaciones eficientes

Objetivo:
- generar ~1M observaciones
- mantener fraude ~1%
- evitar OOM en Kaggle


In [ ]:

# ====================================
# INSTALLS (KAGGLE)
# ====================================

!pip install -q sdv pyarrow fastparquet



In [ ]:

# ====================================
# IMPORTS
# ====================================

import warnings
warnings.filterwarnings("ignore")

import gc
import os
import random

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import roc_auc_score

from sdv.single_table import CTGANSynthesizer
from sdv.single_table import TVAESynthesizer
from sdv.metadata import SingleTableMetadata

from tqdm import tqdm

import pyarrow.parquet as pq
import pyarrow as pa


In [ ]:

# ====================================
# CONFIG
# ====================================

DATASET_PATH = "/kaggle/input/biocatch/biocatch_dataset.csv"

OUTPUT_PARQUET = "biocatch_augmented_1M.parquet"

TARGET_ROWS = 1_000_000
TARGET_VISHING = 10_000
TARGET_LEGIT = TARGET_ROWS - TARGET_VISHING

CHUNK_SIZE = 100_000

RANDOM_STATE = 42

np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)


In [ ]:

# ====================================
# LOAD DATASET
# ====================================

df = pd.read_csv(DATASET_PATH)

print(df.shape)

df.head()


In [ ]:

# ====================================
# MEMORY OPTIMIZATION
# ====================================

def optimize_memory(df):

    for col in df.columns:

        if df[col].dtype == 'float64':
            df[col] = df[col].astype('float32')

        elif df[col].dtype == 'int64':
            df[col] = df[col].astype('int32')

        elif df[col].dtype == 'object':

            nunique = df[col].nunique()

            if nunique / len(df) < 0.5:
                df[col] = df[col].astype('category')

    return df

df = optimize_memory(df)

df.info(memory_usage='deep')


In [ ]:

# ====================================
# REMOVE LEAKAGE
# ====================================

leakage_cols = [
    'biocatch_risk_score',
    'biocatch_genuine_score',
    'biocatch_ato_indicator',
    'biocatch_social_eng_indicator',
    'biocatch_bot_indicator',
    'claim_category',
    'days_to_claim'
]

existing_leakage = [
    c for c in leakage_cols if c in df.columns
]

df_model = df.drop(columns=existing_leakage)

print(df_model.shape)


In [ ]:

# ====================================
# SPLIT CLASSES
# ====================================

legit_df = df_model[df_model['is_vishing'] == 0].copy()
vishing_df = df_model[df_model['is_vishing'] == 1].copy()

print(legit_df.shape)
print(vishing_df.shape)


In [ ]:

# ====================================
# SDV DATA PREPARATION
# ====================================

def prepare_for_sdv(df_input):

    df_sdv = df_input.copy()

    category_cols = df_sdv.select_dtypes(include=['category']).columns

    for col in category_cols:
        df_sdv[col] = df_sdv[col].astype('object')

    return df_sdv



# Training Strategy

Primero validar pipeline con pocos epochs.
Luego aumentar progresivamente.

Esto evita perder horas por:
- OOM
- errores
- configuraciones inválidas


In [ ]:

# ====================================
# SDV METADATA
# ====================================

metadata = SingleTableMetadata()

metadata.detect_from_dataframe(df_model)

metadata


In [ ]:

# ====================================
# FAST CTGAN CONFIG
# ====================================

ctgan_legit = CTGANSynthesizer(
    metadata=metadata,
    epochs=20,
    batch_size=500,
    verbose=True
)

ctgan_vishing = CTGANSynthesizer(
    metadata=metadata,
    epochs=20,
    batch_size=128,
    verbose=True
)


In [ ]:

# ====================================
# TRAIN LEGIT CTGAN
# ====================================

legit_df_sdv = prepare_for_sdv(legit_df)

ctgan_legit.fit(legit_df_sdv)

gc.collect()

print("Legit CTGAN trained")


In [ ]:

# ====================================
# TRAIN VISHING CTGAN
# ====================================

vishing_df_sdv = prepare_for_sdv(vishing_df)

ctgan_vishing.fit(vishing_df_sdv)

gc.collect()

print("Vishing CTGAN trained")


In [ ]:

# ====================================
# TVAE
# ====================================

df_model_sdv = prepare_for_sdv(df_model)

tvae = TVAESynthesizer(
    metadata=metadata,
    epochs=15,
    batch_size=512
)

tvae.fit(df_model_sdv)

gc.collect()

print("TVAE trained")


In [ ]:

# ====================================
# CONSTRAINT ENGINE
# ====================================

def apply_constraints(df_aug):

    if 'phone_call_active' in df_aug.columns and 'call_overlap_duration_s' in df_aug.columns:

        mask = df_aug['phone_call_active'] == 0

        df_aug.loc[mask, 'call_overlap_duration_s'] = 0

    if 'transaction_attempted' in df_aug.columns and 'transaction_amount_cop' in df_aug.columns:

        mask = df_aug['transaction_attempted'] == 0

        df_aug.loc[mask, 'transaction_amount_cop'] = 0

    if 'dead_time_ratio' in df_aug.columns:

        df_aug['dead_time_ratio'] = np.clip(
            df_aug['dead_time_ratio'],
            0,
            1
        )

    if 'segmented_typing_ratio' in df_aug.columns:

        df_aug['segmented_typing_ratio'] = np.clip(
            df_aug['segmented_typing_ratio'],
            0,
            1
        )

    return df_aug


In [ ]:

# ====================================
# HARD CASE GENERATION
# ====================================

def generate_hard_legit(df_input, n):

    hard = df_input.sample(
        n=n,
        replace=True
    ).copy()

    if 'phone_call_active' in hard.columns:
        hard['phone_call_active'] = np.random.binomial(
            1,
            0.35,
            size=n
        )

    if 'hesitation_count' in hard.columns:
        hard['hesitation_count'] *= np.random.uniform(
            1.2,
            1.8,
            size=n
        )

    hard['is_vishing'] = 0

    return hard

def generate_soft_vishing(df_input, n):

    soft = df_input.sample(
        n=n,
        replace=True
    ).copy()

    if 'hesitation_count' in soft.columns:
        soft['hesitation_count'] *= np.random.uniform(
            0.5,
            0.8,
            size=n
        )

    soft['is_vishing'] = 1

    return soft


In [ ]:

# ====================================
# PARQUET STREAMING WRITER
# ====================================

parquet_writer = None

def append_parquet(df_chunk):

    global parquet_writer

    table = pa.Table.from_pandas(df_chunk)

    if parquet_writer is None:

        parquet_writer = pq.ParquetWriter(
            OUTPUT_PARQUET,
            table.schema,
            compression='snappy'
        )

    parquet_writer.write_table(table)



# Chunked Generation

La generación chunked evita:
- RAM overflow
- copies gigantes
- crashes de Kaggle

Cada chunk:
- se genera
- se valida
- se exporta
- se libera de memoria


In [ ]:

# ====================================
# GENERATE LEGIT CHUNKS
# ====================================

remaining_legit = TARGET_LEGIT

while remaining_legit > 0:

    current_chunk = min(CHUNK_SIZE, remaining_legit)

    ctgan_chunk = ctgan_legit.sample(
        num_rows=int(current_chunk * 0.7)
    )

    tvae_chunk = tvae.sample(
        num_rows=int(current_chunk * 0.2)
    )

    bootstrap_chunk = legit_df.sample(
        n=int(current_chunk * 0.1),
        replace=True
    ).copy()

    hard_legit = generate_hard_legit(
        legit_df,
        n=max(1000, int(current_chunk * 0.02))
    )

    chunk = pd.concat([
        ctgan_chunk,
        tvae_chunk,
        bootstrap_chunk,
        hard_legit
    ], ignore_index=True)

    chunk['is_vishing'] = 0

    chunk = apply_constraints(chunk)

    chunk = optimize_memory(chunk)

    append_parquet(chunk)

    remaining_legit -= len(chunk)

    print(f"Remaining legit: {remaining_legit}")

    del ctgan_chunk
    del tvae_chunk
    del bootstrap_chunk
    del hard_legit
    del chunk

    gc.collect()


In [ ]:

# ====================================
# GENERATE VISHING CHUNKS
# ====================================

remaining_vishing = TARGET_VISHING

while remaining_vishing > 0:

    current_chunk = min(20_000, remaining_vishing)

    ctgan_chunk = ctgan_vishing.sample(
        num_rows=int(current_chunk * 0.6)
    )

    tvae_chunk = tvae.sample(
        num_rows=int(current_chunk * 0.2)
    )

    bootstrap_chunk = vishing_df.sample(
        n=int(current_chunk * 0.2),
        replace=True
    ).copy()

    soft_vishing = generate_soft_vishing(
        vishing_df,
        n=max(500, int(current_chunk * 0.05))
    )

    chunk = pd.concat([
        ctgan_chunk,
        tvae_chunk,
        bootstrap_chunk,
        soft_vishing
    ], ignore_index=True)

    chunk['is_vishing'] = 1

    chunk = apply_constraints(chunk)

    chunk = optimize_memory(chunk)

    append_parquet(chunk)

    remaining_vishing -= len(chunk)

    print(f"Remaining vishing: {remaining_vishing}")

    del ctgan_chunk
    del tvae_chunk
    del bootstrap_chunk
    del soft_vishing
    del chunk

    gc.collect()


In [ ]:

# ====================================
# CLOSE WRITER
# ====================================

if parquet_writer:
    parquet_writer.close()

print("Parquet export completed")


In [ ]:

# ====================================
# LOAD FINAL DATASET
# ====================================

final_df = pd.read_parquet(OUTPUT_PARQUET)

print(final_df.shape)

final_df.head()


In [ ]:

# ====================================
# ADVERSARIAL VALIDATION
# ====================================

orig = df_model.sample(
    n=min(50000, len(df_model)),
    random_state=42
).copy()

orig['synthetic'] = 0

synth = final_df.sample(
    n=min(150000, len(final_df)),
    random_state=42
).copy()

synth['synthetic'] = 1

adv_df = pd.concat([orig, synth])

drop_cols = [
    c for c in [
        'session_id',
        'customer_id',
        'session_timestamp'
    ]
    if c in adv_df.columns
]

X = adv_df.drop(columns=['synthetic'] + drop_cols)

for col in X.select_dtypes(include=['object', 'category']).columns:

    le = LabelEncoder()

    X[col] = le.fit_transform(
        X[col].astype(str)
    )

X = X.fillna(0)

y = adv_df['synthetic']

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

clf = RandomForestClassifier(
    n_estimators=50,
    max_depth=10,
    n_jobs=-1,
    random_state=42
)

clf.fit(X_train, y_train)

preds = clf.predict_proba(X_test)[:,1]

auc = roc_auc_score(y_test, preds)

print(f"Adversarial Validation AUC: {auc:.4f}")


In [ ]:

# ====================================
# FINAL STATS
# ====================================

print(final_df['is_vishing'].value_counts())

print(final_df['is_vishing'].value_counts(normalize=True))

print(final_df.memory_usage(deep=True).sum() / 1024**2, "MB")



# Recomendaciones de Ejecución

## Primera corrida

NO generar 1M inicialmente.

Usar:

```python
TARGET_ROWS = 100_000
```

para validar.

---

## Luego

Escalar progresivamente:

- 250k
- 500k
- 1M

---

## Configuración recomendada Kaggle

- GPU P100/T4
- Internet OFF
- RAM High

---

## Tiempo esperado

| Etapa | Tiempo |
|---|---|
| Training | 30-60 min |
| Generation | 10-30 min |
| Validation | 5-15 min |

Tiempo total esperado:

~45 min a 1.5h
